In [7]:
import numpy as np
import heapq
import pandas as pd
from scipy.stats import skewnorm
from sklearn.cluster import KMeans
import os


In [8]:

# =====================================================
# PARAMETERS
# =====================================================

SIM_TIME             = 45 * 24
DRIVER_RATE          = 4.74
RIDER_RATE           = 34.6
PATIENCE_RATE        = 5
SPEED                = 20
AREA_SIZE            = 20
BASE_FARE            = 3
FARE_PER_MILE        = 2
PETROL_COST_PER_MILE = 0.20
CVAR_ALPHA           = 0.05
LAMBDA               = 1.5    # hub weight in scoring formula

OUTPUT_DIR           = "kmeans_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [9]:

# =====================================================
# STEP 1 — BUILD KMEANS HUB CONFIGS FROM REAL DATA
# =====================================================

def parse_coord(s):
    s = str(s).strip('()').split(',')
    return float(s[0]), float(s[1])

print("Loading real data and fitting KMeans hubs...")
riders = pd.read_excel('riders.xlsx')
riders[['px','py']] = pd.DataFrame(
    riders['pickup_location'].apply(parse_coord).tolist(),
    index=riders.index)
X = riders[['px','py']].values

# Fit K=1 to K=10 on all pickup data
HUB_CONFIGS = {}
for k in range(1, 11):
    km     = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X)
    sizes  = np.bincount(km.labels_)
    order  = np.argsort(-sizes)
    HUB_CONFIGS[k] = km.cluster_centers_[order]
    print(f"  K={k}: {[tuple(round(float(c),2) for c in h) for h in HUB_CONFIGS[k]]}")

print(f"\nHub configs ready for K=1 to K=10\n")


Loading real data and fitting KMeans hubs...
  K=1: [(8.36, 12.32)]
  K=2: [(5.51, 10.34), (11.5, 14.51)]
  K=3: [(5.43, 14.86), (12.84, 14.1), (6.78, 7.6)]
  K=4: [(5.61, 15.0), (12.83, 15.84), (11.36, 9.08), (4.54, 7.88)]
  K=5: [(4.13, 13.84), (9.03, 16.16), (10.9, 9.16), (4.73, 6.98), (14.9, 15.05)]
  K=6: [(8.66, 11.62), (3.65, 14.42), (9.06, 16.99), (4.34, 7.08), (14.79, 14.89), (11.98, 7.2)]
  K=7: [(8.77, 11.74), (9.67, 16.93), (4.26, 15.87), (3.39, 10.3), (14.97, 15.24), (6.07, 5.44), (12.99, 8.18)]
  K=8: [(8.21, 11.43), (9.73, 16.5), (4.44, 16.3), (3.23, 11.23), (13.89, 11.18), (4.39, 5.87), (10.64, 6.13), (15.11, 16.63)]
  K=9: [(7.1, 11.51), (9.46, 17.0), (11.1, 12.41), (4.31, 16.36), (2.79, 11.35), (9.63, 6.32), (15.16, 16.34), (3.95, 5.91), (15.12, 9.42)]
  K=10: [(11.09, 12.35), (6.92, 13.65), (9.71, 17.33), (7.35, 9.35), (2.84, 11.24), (3.85, 16.8), (15.18, 16.44), (3.79, 5.45), (10.56, 5.64), (15.33, 10.01)]

Hub configs ready for K=1 to K=10



In [10]:

# =====================================================
# LOCATION FUNCTIONS
# =====================================================

def driver_initial_location():
    x = np.clip(np.random.normal(9.97,  4.36), 0, AREA_SIZE)
    y = np.clip(np.random.normal(11.51, 4.34), 0, AREA_SIZE)
    return np.array([x, y])

def rider_pickup_location():
    x = np.clip(skewnorm.rvs(a= 1.80, loc= 4.18, scale=5.97), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-2.49, loc=17.05, scale=6.31), 0, AREA_SIZE)
    return np.array([x, y])

def rider_dropoff_location():
    x = np.clip(skewnorm.rvs(a= -1.65, loc=15.51, scale=6.24), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-14.26, loc=19.50, scale=7.50), 0, AREA_SIZE)
    return np.array([x, y])


In [11]:

# =====================================================
# DRIVER & RIDER CLASSES
# =====================================================

class Driver:
    def __init__(self, driver_id, location, online_until):
        self.id           = driver_id
        self.location     = location
        self.online_until = online_until
        self.earnings     = 0
        self.status       = "idle"
        self.busy_time    = 0

class Rider:
    def __init__(self, rider_id, origin, destination, arrival_time, abandon_time):
        self.id           = rider_id
        self.origin       = origin
        self.destination  = destination
        self.arrival_time = arrival_time
        self.abandon_time = abandon_time
        self.matched      = False


In [12]:

# =====================================================
# SIMULATION — Idea 3b (dropoff hub distance scoring)
# =====================================================

class Simulation:

    def __init__(self, hubs, lam=LAMBDA):
        """
        hubs : np.array of hub coordinates (from KMeans)
        lam  : weight on hub distance in matching score
        """
        self.hubs = hubs
        self.lam  = lam

        self.current_time     = 0
        self.event_list       = []
        self.idle_drivers     = {}
        self.busy_drivers     = {}
        self.waiting_riders   = {}
        self.driver_counter   = 0
        self.rider_counter    = 0
        self.total_riders     = 0
        self.abandoned_riders = 0
        self.total_queue_wait = 0
        self.total_true_wait  = 0
        self.driver_log       = []

    def exp_time(self, rate):
        return np.random.exponential(1 / rate)

    def dist(self, a, b):
        return np.linalg.norm(np.array(a) - np.array(b))

    def travel_time(self, d):
        mu = d / SPEED
        return np.random.uniform(0.8 * mu, 1.2 * mu)

    def schedule(self, t, e, d=None):
        heapq.heappush(self.event_list, (t, e, d))

    def nearest_hub_dist(self, location):
        return min(np.linalg.norm(location - h) for h in self.hubs)

    # ── Matching: score = dist(driver→pickup) + λ × dist(dropoff→hub) ──

    def select_rider(self, driver):
        def score(rid):
            rider        = self.waiting_riders[rid]
            d_to_pickup  = self.dist(driver.location, rider.origin)
            dropoff_hub  = self.nearest_hub_dist(rider.destination)
            return d_to_pickup + self.lam * dropoff_hub
        return min(self.waiting_riders, key=score)

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders:
            return

        driver_id = next(iter(self.idle_drivers))
        driver    = self.idle_drivers[driver_id]
        rider_id  = self.select_rider(driver)

        rider        = self.waiting_riders[rider_id]
        rider.matched = True

        pickup_dist  = self.dist(driver.location, rider.origin)
        trip_dist    = self.dist(rider.origin, rider.destination)
        pickup_time  = self.travel_time(pickup_dist)
        trip_time    = self.travel_time(trip_dist)
        tt           = pickup_time + trip_time

        queue_wait            = self.current_time - rider.arrival_time
        self.total_queue_wait += queue_wait
        self.total_true_wait  += queue_wait + pickup_time

        earnings = (BASE_FARE + FARE_PER_MILE * trip_dist) - \
                   PETROL_COST_PER_MILE * (pickup_dist + trip_dist)

        driver.status     = "busy"
        driver.busy_time += tt
        self.busy_drivers[driver_id] = driver
        del self.idle_drivers[driver_id]
        del self.waiting_riders[rider_id]

        self.schedule(self.current_time + tt, "DROPOFF",
                      (driver_id, rider.destination, earnings))

    # ── Event handlers ──────────────────────────────────────

    def handle_driver_arrival(self):
        self.driver_counter += 1
        did          = self.driver_counter
        loc          = driver_initial_location()
        online_until = self.current_time + np.random.uniform(6, 8)
        driver       = Driver(did, loc, online_until)
        self.idle_drivers[did] = driver
        self.driver_log.append({
            "driver_id"   : did,
            "arrival_time": self.current_time,
            "online_until": online_until,
            "earnings"    : 0,
            "busy_time"   : 0,
        })
        self.schedule(online_until, "DRIVER_OFFLINE", did)
        self.schedule(self.current_time + self.exp_time(DRIVER_RATE), "DRIVER_ARRIVAL")
        self.try_match()

    def handle_rider_arrival(self):
        self.rider_counter += 1
        rid          = self.rider_counter
        self.total_riders += 1
        origin       = rider_pickup_location()
        dest         = rider_dropoff_location()
        abandon_time = self.current_time + self.exp_time(PATIENCE_RATE)
        self.waiting_riders[rid] = Rider(rid, origin, dest,
                                         self.current_time, abandon_time)
        self.schedule(abandon_time, "RIDER_ABANDON", rid)
        self.schedule(self.current_time + self.exp_time(RIDER_RATE), "RIDER_ARRIVAL")
        self.try_match()

    def handle_dropoff(self, data):
        did, new_loc, earnings = data
        driver = self.busy_drivers[did]
        driver.earnings += earnings
        driver.location  = new_loc
        driver.status    = "idle"
        for d in self.driver_log:
            if d["driver_id"] == did:
                d["earnings"]  += earnings
                d["busy_time"]  = driver.busy_time
                break
        del self.busy_drivers[did]
        if self.current_time < driver.online_until:
            self.idle_drivers[did] = driver
        self.try_match()

    def run(self):
        self.schedule(0, "DRIVER_ARRIVAL")
        self.schedule(0, "RIDER_ARRIVAL")
        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, et, data = heapq.heappop(self.event_list)
            if   et == "DRIVER_ARRIVAL": self.handle_driver_arrival()
            elif et == "RIDER_ARRIVAL":  self.handle_rider_arrival()
            elif et == "DROPOFF":        self.handle_dropoff(data)
            elif et == "RIDER_ABANDON":
                if data in self.waiting_riders:
                    self.abandoned_riders += 1
                    del self.waiting_riders[data]
            elif et == "DRIVER_OFFLINE":
                self.idle_drivers.pop(data, None)
        return self.collect_results()

    def collect_results(self):
        matched = self.total_riders - self.abandoned_riders
        df      = pd.DataFrame(self.driver_log)
        df["online_duration"]  = df["online_until"] - df["arrival_time"]
        df["busy_time_capped"] = df[["busy_time","online_duration"]].min(axis=1)
        df["idle_time"]        = df["online_duration"] - df["busy_time_capped"]

        p  = df["earnings"].values
        N  = len(p);  mp = p.mean()
        vp = np.sum((p - mp)**2) / (N - 1) if N > 1 else 0
        ps = np.sort(p)
        cvar   = np.mean(ps[:max(1, int(np.floor(CVAR_ALPHA * N)))])
        cvar_m = np.mean(ps[:max(1, int(np.floor(0.5 * N)))])

        return {
            # Rider metrics
            "total_riders"     : self.total_riders,
            "matched_riders"   : matched,
            "abandoned_riders" : self.abandoned_riders,
            "abandonment_rate" : self.abandoned_riders / self.total_riders,
            "avg_queue_wait"   : self.total_queue_wait / matched if matched > 0 else 0,
            "avg_true_wait"    : self.total_true_wait  / matched if matched > 0 else 0,
            # Driver metrics
            "total_drivers"    : N,
            "avg_earnings"     : mp,
            "avg_profit_per_hr": df["earnings"].sum() / df["online_duration"].sum(),
            "avg_idle_time"    : df["idle_time"].mean(),
            # Fairness
            "variance"         : vp,
            "fairness_cv"      : np.sqrt(vp) / mp if mp > 0 else 0,
            "cvar"             : cvar,
            "delta_cvar"       : cvar - cvar_m,
            # Raw profits for cross-run fairness
            "_profits"         : p,
            "_driver_df"       : df,
        }


In [13]:

# =====================================================
# RUN BASELINE + K=1 TO K=10
# =====================================================

all_results   = {}
summary_rows  = []

# ── Baseline (no hubs — pure closest distance) ────────
print("=" * 55)
print("Running: Baseline (no hubs)...")

class BaselineSim(Simulation):
    def select_rider(self, driver):
        return min(self.waiting_riders,
                   key=lambda r: self.dist(driver.location,
                                           self.waiting_riders[r].origin))

np.random.seed(42)
baseline_hubs = np.array([[0,0]])   # dummy, not used
sim           = BaselineSim(hubs=baseline_hubs, lam=0)
res           = sim.run()
all_results["baseline"] = res

# Save baseline driver data
res["_driver_df"].to_csv(
    f"{OUTPUT_DIR}/data_baseline.csv", index=False)

print(f"  Abandon: {res['abandonment_rate']*100:.2f}%  "
      f"QueueWait: {res['avg_queue_wait']*60:.3f} min  "
      f"TrueWait: {res['avg_true_wait']*60:.3f} min  "
      f"Earn: £{res['avg_earnings']:.2f}")

summary_rows.append({
    "config"          : "Baseline",
    "n_hubs"          : 0,
    "abandonment_pct" : res["abandonment_rate"] * 100,
    "avg_queue_wait_min": res["avg_queue_wait"] * 60,
    "avg_true_wait_min" : res["avg_true_wait"]  * 60,
    "matched_riders"  : res["matched_riders"],
    "avg_earnings"    : res["avg_earnings"],
    "avg_profit_per_hr": res["avg_profit_per_hr"],
    "avg_idle_time"   : res["avg_idle_time"],
    "variance"        : res["variance"],
    "fairness_cv"     : res["fairness_cv"],
    "cvar"            : res["cvar"],
    "delta_cvar"      : res["delta_cvar"],
    "fairness_effect" : 0.0,
    "delta_cvar_cross": 0.0,
    "delta_profit"    : 0.0,
})

Running: Baseline (no hubs)...
  Abandon: 3.44%  QueueWait: 0.306 min  TrueWait: 24.246 min  Earn: £111.77


In [ ]:

# ── K=1 to K=10 ───────────────────────────────────────
for k in range(1, 11):
    label = f"kmean_{k}"
    print(f"Running: K={k} hubs...")

    np.random.seed(42)
    sim = Simulation(hubs=HUB_CONFIGS[k], lam=LAMBDA)
    res = sim.run()
    all_results[label] = res

    # Save individual driver-level CSV
    res["_driver_df"].to_csv(
        f"{OUTPUT_DIR}/data_{label}.csv", index=False)

    # Cross-run fairness vs baseline
    b              = all_results["baseline"]
    fe             = 1 - res["variance"] / b["variance"] if b["variance"] > 0 else 0
    delta_cvar_x   = res["cvar"] - b["cvar"]
    delta_profit   = res["avg_earnings"] - b["avg_earnings"]

    print(f"  Abandon: {res['abandonment_rate']*100:.2f}%  "
          f"QueueWait: {res['avg_queue_wait']*60:.3f} min  "
          f"TrueWait: {res['avg_true_wait']*60:.3f} min  "
          f"Earn: £{res['avg_earnings']:.2f}  "
          f"FairnessEff: {fe:+.4f}")

    summary_rows.append({
        "config"           : label,
        "n_hubs"           : k,
        "abandonment_pct"  : res["abandonment_rate"] * 100,
        "avg_queue_wait_min": res["avg_queue_wait"] * 60,
        "avg_true_wait_min" : res["avg_true_wait"]  * 60,
        "matched_riders"   : res["matched_riders"],
        "avg_earnings"     : res["avg_earnings"],
        "avg_profit_per_hr": res["avg_profit_per_hr"],
        "avg_idle_time"    : res["avg_idle_time"],
        "variance"         : res["variance"],
        "fairness_cv"      : res["fairness_cv"],
        "cvar"             : res["cvar"],
        "delta_cvar"       : res["delta_cvar"],
        "fairness_effect"  : fe,
        "delta_cvar_cross" : delta_cvar_x,
        "delta_profit"     : delta_profit,
    })

# =====================================================
# SAVE SUMMARY TABLE
# =====================================================

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/summary_all_kmeans.csv", index=False)

# =====================================================
# PRINT SUMMARY TABLE
# =====================================================

b = all_results["baseline"]

print("\n" + "=" * 100)
print("  PERFORMANCE METRICS")
print("=" * 100)
print(f"  {'Config':<14} {'Hubs':>5} {'Abandon%':>9} {'Queue(min)':>11} "
      f"{'True(min)':>10} {'AvgEarn£':>9} {'£/hr':>7} {'CVaR£':>8} {'ΔAbdn':>9}")
print("-" * 100)
for row in summary_rows:
    da   = row["abandonment_pct"] - summary_rows[0]["abandonment_pct"]
    flag = f"{'↓' if da<0 else '↑'}{abs(da):.2f}pp" if row["config"] != "Baseline" else "—"
    print(f"  {row['config']:<14} {row['n_hubs']:>5} "
          f"{row['abandonment_pct']:>8.2f}% "
          f"{row['avg_queue_wait_min']:>11.3f} "
          f"{row['avg_true_wait_min']:>10.3f} "
          f"{row['avg_earnings']:>9.2f} "
          f"{row['avg_profit_per_hr']:>7.2f} "
          f"{row['cvar']:>8.2f} "
          f"{flag:>9}")
print("=" * 100)

print("\n" + "=" * 80)
print("  FAIRNESS METRICS  (pre = Baseline, post = KMeans config)")
print("=" * 80)
print(f"  {'Config':<14} {'Var(pre)':>10} {'Var(post)':>10} "
      f"{'Δprofit£':>10} {'FairnessEff':>12} {'ΔCVaR£':>9}")
print("-" * 80)
for row in summary_rows:
    if row["config"] == "Baseline":
        print(f"  {row['config']:<14} {row['variance']:>10.2f} "
              f"{'—':>10} {'—':>10} {'—':>12} {'—':>9}")
    else:
        print(f"  {row['config']:<14} {summary_rows[0]['variance']:>10.2f} "
              f"{row['variance']:>10.2f} "
              f"{row['delta_profit']:>+10.2f} "
              f"{row['fairness_effect']:>+12.4f} "
              f"{row['delta_cvar_cross']:>+9.2f}")
print("=" * 80)

print(f"\nFiles saved to: {OUTPUT_DIR}/")
print(f"  data_baseline.csv")
for k in range(1, 11):
    print(f"  data_kmean_{k}.csv")
print(f"  summary_all_kmeans.csv")